# 11 â€” Symptom text and fusion comparison

**In short:** the chosen image model (ResNet-50, notebook 09) is **frozen**, and symptom text
is added beside it. Several text encoders and fusion methods are compared in stages, the
same way the image architectures were, and one pre-set rule picks the winner.

| Stage | What is compared | Held fixed |
|---|---|---|
| 1 | Text encoder: `tfidf`, `distilbert`, `bert_base`, `minilm`, `mpnet`, `scibert` | concat fusion, 20% missing text |
| 2 | Fusion: `late`, `concat`, `gated` | Stage 1 winner, 20% missing text |
| 3 | Missing-text rate: 10, 20, 30, 50% | Stage 1 and 2 winners |
| 4 | One check: fine-tuning ResNet-50 together with the text (B) instead of freezing it (A) | the final setup |

Every setup is trained with **3 seeds**; the mean and spread are reported.

**Winning rule, fixed before any results** (decisions D4/D5):
1. *Requirement (Hypothesis 1):* with **no text**, macro-F1 on the field validation photos must be
   no worse than ResNet-50 alone, minus 0.5 points. Setups that fail are out.
2. Among the rest, the lowest **log-loss with text** on the field validation photos wins
   (diseased photos with a validation sentence of their class where one exists; healthy photos
   without text, because no source describes healthy leaves).
3. If the best and others overlap within one standard deviation across seeds, it is a tie:
   the Chapter 3 default is kept (DistilBERT, concatenation); for the missing-text rate,
   the **higher** rate is kept.
4. The fine-tuning check (B) is adopted only if it meets the requirement **and** beats A by
   more than the seed spread.

Also reported for every setup: the three **healthy-bias tests** (does text push healthy photos
towards a disease; are diseased photos without text called healthy). PlantDoc is reported at
the end but **never** used to choose. CPU time per sentence is measured for each encoder.

**Needs a GPU runtime (T4).** Image features are computed once (about 15â€“20 minutes) and
saved to Drive; after that every run takes seconds to minutes.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings

`QUICK_TEST = True` runs everything on a small sample with one seed, to check the notebook
end to end in a few minutes. Set it to `False` for the real comparison.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import torch

from src import fusion, metrics, text_encoders, training
for module in (fusion, metrics, text_encoders, training):
    importlib.reload(module)

QUICK_TEST = True

IMAGE_RUN = "compare_resnet50"          # the chosen image model (notebook 09/10)
ENCODERS = list(text_encoders.ENCODERS)  # Stage 1 candidates
SEEDS = [42] if QUICK_TEST else [42, 43, 44]
DEFAULT_ENCODER, DEFAULT_FUSION = "distilbert", "concat"  # Chapter 3 defaults, kept on a tie
STAGE_RATE = 0.2                          # missing-text rate during Stages 1 and 2
H1_MARGIN = 0.005                         # "no worse than ResNet-50 alone" within 0.5 points
FIELD_SOURCES = ["fieldplant", "tanzania"]

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"
OUT_DIR = config.MODELS_DIR / ("fusion_quick_test" if QUICK_TEST else "fusion")
FEATURES_DIR = config.MODELS_DIR / "fusion_features"  # shared: computing them is the slow step
OUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
training.seed_everything(config.RANDOM_SEED)
print("Output:", OUT_DIR, "| GPU:", torch.cuda.get_device_name(0))

## Load the split and the reviewed symptom sentences

The image split is the same as in notebooks 08â€“10. The sentences are the hand-reviewed rows
(`keep == y`) of `ml/data/symptoms/descriptions.csv`; their split is by source document.

In [ ]:
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in ["train", "val", "test", "ood_test"]}
CLASSES = list(config.PLANTVILLAGE_CLASSES)
if QUICK_TEST:
    splits = {name: rows.groupby("class", group_keys=False).head(20) for name, rows in splits.items()}

descriptions = pd.read_csv(ML_DIR / "data" / "symptoms" / "descriptions.csv", encoding="utf-8-sig")
descriptions = descriptions[descriptions["keep"] == "y"].reset_index(drop=True)
display(pd.DataFrame({name: rows["source"].value_counts() for name, rows in splits.items()}).fillna(0).astype(int))
display(descriptions.pivot_table(index="class_name", columns="split", values="text", aggfunc="count").fillna(0).astype(int))

val_field_rows = splits["val"][splits["val"]["source"].isin(FIELD_SOURCES)]
print("Field validation photos per class (selection set):")
display(val_field_rows["class"].value_counts())

## Step 1 â€” image features, once

ResNet-50 is loaded from its best epoch and frozen. For every photo we keep its 2048-number
summary (the input of its output layer) and its own 11 scores. Training photos use the
evaluation transform: no flips or crops at this stage, because the features are computed once.

In [ ]:
import shutil

from torch import nn
from torch.utils.data import DataLoader

from src.data import image_dataset

LOCAL_ROOT = Path("/content/image_cache")
resnet = training.build_model(len(CLASSES), "resnet50", pretrained=False)
resnet.load_state_dict(torch.load(config.MODELS_DIR / IMAGE_RUN / "best.pt", map_location=DEVICE))
output_layer = resnet.fc
resnet.fc = nn.Identity()          # the network now returns the 2048-number summary
resnet.to(DEVICE).eval()
output_layer.to(DEVICE)

@torch.no_grad()
def image_features(name, rows):
    path = FEATURES_DIR / f"{name}{'_quick' if QUICK_TEST else ''}.npz"
    if path.exists():
        saved = np.load(path)
        return fusion.Features(saved["image"], saved["logits"], saved["labels"])
    image_dataset.cache_images(sorted(rows["path"]), config.RAW_DATA_DIR, LOCAL_ROOT)
    dataset = image_dataset.SplitDataset(rows, LOCAL_ROOT, CLASSES, image_dataset.eval_transform(224))
    summaries, scores, labels = [], [], []
    for images, y in DataLoader(dataset, batch_size=128, num_workers=2):
        with torch.autocast(device_type="cuda"):
            summary = resnet(images.to(DEVICE))
            logits = output_layer(summary)
        summaries.append(summary.float().cpu().numpy()); scores.append(logits.float().cpu().numpy()); labels.append(y.numpy())
    feats = fusion.Features(np.concatenate(summaries), np.concatenate(scores), np.concatenate(labels))
    np.savez(path, image=feats.image, logits=feats.logits, labels=feats.labels)
    return feats

feats = {name: image_features(name, rows.reset_index(drop=True)) for name, rows in splits.items()}

def field_part(name):
    mask = splits[name]["source"].isin(FIELD_SOURCES).to_numpy()
    f = feats[name]
    return fusion.Features(f.image[mask], f.logits[mask], f.labels[mask])

val_field = field_part("val")
RESNET = metrics.summarise(val_field.logits, val_field.labels, len(CLASSES))
print({name: f.image.shape for name, f in feats.items()})
print(f"ResNet-50 alone on field validation: macro-F1 {RESNET['macro_f1']:.4f}, log-loss {RESNET['nll']:.4f}")
if not QUICK_TEST:
    print("This should match the val: field row of ml/results/compare_resnet50/metrics.csv.")

## Step 2 â€” sentence vectors for every encoder, once

Each encoder is frozen; TF-IDF learns its vocabulary from the training sentences only.

In [ ]:
texts = descriptions["text"].tolist()
train_texts = descriptions.loc[descriptions["split"] == "train", "text"].tolist()
banks = {}
for name in ENCODERS:
    vectors = text_encoders.encode(name, texts, train_texts, DEVICE)
    banks[name] = {split: fusion.TextBank.build(descriptions, vectors, CLASSES, split) for split in ["train", "val", "test"]}
    print(f"{name:11s} vector size {vectors.shape[1]}")
    torch.cuda.empty_cache()

## The run and the selection rule

`run` trains one setup with one seed and scores it on the field validation photos.
`choose` applies the pre-set rule to a stage's table.

In [ ]:
def no_text(features, bank):
    return np.zeros((len(features.labels), bank.dim), np.float32), np.zeros(len(features.labels), np.float32)

def run(encoder, kind, rate, seed):
    bank = banks[encoder]
    head = fusion.train_head(kind, feats["train"], val_field, bank["train"], bank["val"], CLASSES, rate, seed, DEVICE)
    without = metrics.summarise(fusion.predict(kind, head, val_field, *no_text(val_field, bank["val"]), CLASSES, DEVICE),
                                val_field.labels, len(CLASSES))
    with_text = metrics.summarise(fusion.predict(kind, head, val_field, *bank["val"].fixed(val_field.labels), CLASSES, DEVICE),
                                  val_field.labels, len(CLASSES))
    return head, {"encoder": encoder, "fusion": kind, "missing_rate": rate, "seed": seed,
                  "f1_no_text": without["macro_f1"], "f1_with_text": with_text["macro_f1"],
                  "nll_with_text": with_text["nll"], "ece_with_text": with_text["ece"],
                  **fusion.bias_tests(kind, head, val_field, bank["val"], CLASSES, DEVICE)}

def choose(table, key, default=None, prefer_high=False):
    summary = table.groupby(key).agg(f1_no_text=("f1_no_text", "mean"), nll=("nll_with_text", "mean"),
                                     nll_sd=("nll_with_text", "std")).fillna({"nll_sd": 0.0})
    summary["meets_H1"] = summary["f1_no_text"] >= RESNET["macro_f1"] - H1_MARGIN
    eligible = summary[summary["meets_H1"]]
    if eligible.empty:
        display(summary)
        print("No setup meets the requirement: text must not be used; report image-only.")
        return None, summary
    best = eligible["nll"].idxmin()
    tied = eligible[eligible["nll"] - eligible["nll_sd"] <= eligible.loc[best, "nll"] + eligible.loc[best, "nll_sd"]].index.tolist()
    if len(tied) > 1 and prefer_high:
        winner, reason = max(tied), f"tie between {tied}: higher rate kept"
    elif len(tied) > 1 and default in tied:
        winner, reason = default, f"tie between {tied}: Chapter 3 default kept"
    else:
        winner, reason = best, "lowest log-loss with text" + (f" (tie {tied}, default not among them)" if len(tied) > 1 else "")
    summary["chosen"] = summary.index == winner
    display(summary.round(4))
    print(f"Chosen {key}: {winner} â€” {reason}")
    return winner, summary

## Stage 1 â€” text encoder

Concatenation fusion and 20% missing text for every encoder; only the encoder changes.

In [ ]:
stage1 = pd.DataFrame([run(enc, DEFAULT_FUSION, STAGE_RATE, seed)[1] for enc in ENCODERS for seed in SEEDS])
ENCODER, stage1_summary = choose(stage1, "encoder", default=DEFAULT_ENCODER)
stage1.to_csv(OUT_DIR / "stage1_encoders.csv", index=False)

## Stage 2 â€” fusion method

The Stage 1 encoder with late fusion, concatenation and the gated head.

In [ ]:
assert ENCODER is not None, "Stage 1 chose no encoder; stop here and report image-only."
stage2 = pd.DataFrame([run(ENCODER, kind, STAGE_RATE, seed)[1] for kind in fusion.FUSIONS for seed in SEEDS])
FUSION, stage2_summary = choose(stage2, "fusion", default=DEFAULT_FUSION)
stage2.to_csv(OUT_DIR / "stage2_fusion.csv", index=False)

## Stage 3 â€” missing-text rate

Chapter 3 compares 10, 20, 30 and 50%. Higher rates also weaken the "no text means healthy"
shortcut, so the bias tests are shown beside the log-loss; on a tie the higher rate is kept.

In [ ]:
assert FUSION is not None
stage3 = pd.DataFrame([run(ENCODER, FUSION, rate, seed)[1] for rate in fusion.MISSING_RATES for seed in SEEDS])
RATE, stage3_summary = choose(stage3, "missing_rate", prefer_high=True)
bias_columns = [c for c in stage3.columns if c.startswith(("healthy_", "diseased_"))]
display(stage3.groupby("missing_rate")[bias_columns].mean().round(3))
stage3.to_csv(OUT_DIR / "stage3_missing_rate.csv", index=False)

## Stage 4 â€” check: fine-tune ResNet-50 together with the text (B)

So far ResNet-50 was frozen (A). Here the whole network trains with the chosen head, starting
from A, with a small learning rate for ResNet-50 (1e-5) and the chosen missing-text rate, for
5 epochs with the usual flips and crops. B is adopted only if it meets the requirement and
beats A by more than the seed spread. Late fusion trains nothing jointly, so if it won, this
check does not apply. About 15â€“20 minutes per seed.

In [ ]:
from src.data import image_dataset

def fine_tune(head, seed, epochs=1 if QUICK_TEST else 5):
    # Features may have come from Drive in this session, so make sure the photos are on local disk.
    image_dataset.cache_images(sorted(set(pd.concat([splits["train"], splits["val"], splits["test"], splits["ood_test"]])["path"])),
                               config.RAW_DATA_DIR, LOCAL_ROOT)
    training.seed_everything(seed)
    rng = np.random.default_rng(seed)
    bank = banks[ENCODER]
    backbone = training.build_model(len(CLASSES), "resnet50", pretrained=False)
    backbone.load_state_dict(torch.load(config.MODELS_DIR / IMAGE_RUN / "best.pt", map_location=DEVICE))
    backbone.fc = nn.Identity()
    backbone.to(DEVICE); head.to(DEVICE)
    train_ds = image_dataset.SplitDataset(splits["train"].reset_index(drop=True), LOCAL_ROOT, CLASSES, image_dataset.train_transform(224))
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True, num_workers=2)
    optimizer = torch.optim.AdamW([{"params": backbone.parameters(), "lr": 1e-5},
                                   {"params": head.parameters(), "lr": 1e-4}], weight_decay=1e-2)
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(epochs):
        backbone.train(); head.train()
        for images, labels in loader:
            text, has = bank["train"].sample(labels.numpy(), RATE, rng)
            with torch.autocast(device_type="cuda"):
                logits = head(backbone(images.to(DEVICE)), torch.as_tensor(text, device=DEVICE), torch.as_tensor(has, device=DEVICE))
                loss = nn.functional.cross_entropy(logits, labels.to(DEVICE))
            optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        print(f"seed {seed} epoch {epoch + 1}/{epochs} done")
    return backbone.eval(), head.eval()

@torch.no_grad()
def features_from(backbone, name):
    rows = splits[name].reset_index(drop=True)
    dataset = image_dataset.SplitDataset(rows, LOCAL_ROOT, CLASSES, image_dataset.eval_transform(224))
    summaries = []
    for images, _ in DataLoader(dataset, batch_size=128, num_workers=2):
        with torch.autocast(device_type="cuda"):
            summaries.append(backbone(images.to(DEVICE)).float().cpu().numpy())
    return fusion.Features(np.concatenate(summaries), feats[name].logits, feats[name].labels)

b_rows = []
if FUSION == "late":
    print("Late fusion won: nothing is trained jointly, so the fine-tuning check does not apply.")
else:
    for seed in SEEDS:
        head_a, row_a = run(ENCODER, FUSION, RATE, seed)
        backbone, head_b = fine_tune(head_a, seed)
        val_b = features_from(backbone, "val")
        mask = splits["val"]["source"].isin(FIELD_SOURCES).to_numpy()
        val_b = fusion.Features(val_b.image[mask], val_b.logits[mask], val_b.labels[mask])
        bank = banks[ENCODER]["val"]
        without = metrics.summarise(fusion.predict(FUSION, head_b, val_b, *no_text(val_b, bank), CLASSES, DEVICE), val_b.labels, len(CLASSES))
        with_text = metrics.summarise(fusion.predict(FUSION, head_b, val_b, *bank.fixed(val_b.labels), CLASSES, DEVICE), val_b.labels, len(CLASSES))
        b_rows += [{**row_a, "variant": "A frozen"},
                   {"seed": seed, "variant": "B fine-tuned", "f1_no_text": without["macro_f1"], "f1_with_text": with_text["macro_f1"],
                    "nll_with_text": with_text["nll"], "ece_with_text": with_text["ece"]}]
        torch.save({"backbone": backbone.state_dict(), "head": head_b.state_dict()}, OUT_DIR / f"b_check_seed{seed}.pt")
    b_check = pd.DataFrame(b_rows)
    b_summary = b_check.groupby("variant").agg(f1_no_text=("f1_no_text", "mean"), nll=("nll_with_text", "mean"),
                                                 nll_sd=("nll_with_text", "std")).fillna(0.0)
    display(b_summary.round(4))
    a, b = b_summary.loc["A frozen"], b_summary.loc["B fine-tuned"]
    ADOPT_B = bool(b["f1_no_text"] >= RESNET["macro_f1"] - H1_MARGIN and a["nll"] - b["nll"] > max(a["nll_sd"], b["nll_sd"]))
    print("Adopt B (fine-tuned)?", ADOPT_B, "â€” otherwise A (frozen) is kept and B is reported as the check.")
    b_check.to_csv(OUT_DIR / "stage4_b_check.csv", index=False)

## Final model and reporting

The chosen setup (A, unless B was adopted) is trained once more with the first seed and scored
on the test photos and PlantDoc. Two conditions, as planned in Chapter 3: **image only** (no
text, the realistic case) and **with text** (diseased test photos paired with a held-out test
sentence: the controlled upper bound). PlantDoc is reported, not used to choose.

In [ ]:
import json

final_head, _ = run(ENCODER, FUSION, RATE, SEEDS[0])
if globals().get("ADOPT_B"):
    # B was adopted: use the fine-tuned network (first seed) for the photos and its head.
    saved = torch.load(OUT_DIR / f"b_check_seed{SEEDS[0]}.pt", map_location=DEVICE)
    backbone = training.build_model(len(CLASSES), "resnet50", pretrained=False); backbone.fc = nn.Identity()
    backbone.load_state_dict(saved["backbone"]); backbone.to(DEVICE).eval()
    final_head.load_state_dict(saved["head"])
    feats = {**feats, "test": features_from(backbone, "test"), "ood_test": features_from(backbone, "ood_test")}
bank = banks[ENCODER]["test"]
rows = []
for name in ["test", "ood_test"]:
    f = feats[name]
    for label, part in [(name, f), (f"{name}: field", field_part(name) if name == "test" else None)]:
        if part is None or len(part.labels) == 0:
            continue
        rows.append({"set": label, "condition": "ResNet-50 alone", **metrics.summarise(part.logits, part.labels, len(CLASSES))})
        rows.append({"set": label, "condition": "fusion, no text",
                     **metrics.summarise(fusion.predict(FUSION, final_head, part, *no_text(part, bank), CLASSES, DEVICE), part.labels, len(CLASSES))})
        rows.append({"set": label, "condition": "fusion, with test text",
                     **metrics.summarise(fusion.predict(FUSION, final_head, part, *bank.fixed(part.labels), CLASSES, DEVICE), part.labels, len(CLASSES))})
final = pd.DataFrame(rows).set_index(["set", "condition"])
display(final.round(4))
final.to_csv(OUT_DIR / "final_metrics.csv")

test_bias = fusion.bias_tests(FUSION, final_head, field_part("test"), bank, CLASSES, DEVICE)
display(pd.Series(test_bias).round(3))

cpu_ms = {name: text_encoders.cpu_milliseconds_per_sentence(name, descriptions["text"].iloc[0]) for name in ENCODERS}
display(pd.Series(cpu_ms, name="CPU ms per sentence (Colab, 2 threads)").round(1))

choice = {"encoder": ENCODER, "fusion": FUSION, "missing_rate": RATE, "variant": "B" if globals().get("ADOPT_B") else "A",
          "resnet50_val_field": RESNET, "test_bias": test_bias, "cpu_ms_per_sentence": cpu_ms, "quick_test": QUICK_TEST}
training.save_json(choice, OUT_DIR / "choice.json")
torch.save(final_head.state_dict(), OUT_DIR / "final_head.pt")
print(json.dumps({k: choice[k] for k in ["encoder", "fusion", "missing_rate", "variant"]}))

## Copy the results into the repository

Weights and features stay on Drive; the small result files go to `ml/results/<run>/`.

In [ ]:
RUN_NAME = OUT_DIR.name
REPO_RESULTS = ML_DIR / "results" / RUN_NAME
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

## Make sure Drive has saved everything

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S). A copy with its outputs goes into the results folder, and only
that folder is committed to a new branch. Skip this cell for the quick test.

In [ ]:
from google.colab import userdata

from src import github_sync

assert not QUICK_TEST, "Quick test: nothing to save. Set QUICK_TEST = False for the real run."
shutil.copy(ML_DIR / "notebooks" / "11_text_fusion_comparison.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/fusion-comparison",
    message="Compare text encoders, fusion methods and missing-text rates",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))